# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import sys
import json
import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient
from openai import OpenAI
from dotenv import load_dotenv
sys.path.append(os.path.abspath('..'))


/Users/vinaym/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
load_dotenv(dotenv_path='../.env')
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print("Credentials loaded.")


Credentials loaded.


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
chroma_client = chromadb.PersistentClient(path="games_chromadb")
openai_key = os.getenv("OPENAI_API_KEY")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=openai_key,
    model_name="text-embedding-3-small",
    api_base="https://openai.vocareum.com/v1/"
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry. 
    """
    res = collection.query(query_texts=[query], n_results=3)
    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    results_list = []
    for i in range(len(docs)):
        m = metas[i] if i < len(metas) else {}
        results_list.append({
            "Platform": m.get("Platform", "Unknown"),
            "Name": m.get("Name", "Unknown"),
            "YearOfRelease": m.get("YearOfRelease", 0),
            "Description": m.get("Description", "")
        })
    return results_list

test_retrieval = retrieve_game("FIFA 21")
print(f"Retrieved {len(test_retrieval)} games.")


Retrieved 3 games.


#### Evaluate Retrieval Tool

In [5]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openai.vocareum.com/v1/"
)

def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 
    args: 
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    """
    context = "\n".join([f"- {d.get('Name')} ({d.get('Platform')}): {d.get('Description')}" for d in retrieved_docs])
    
    system_message = (
        "You are an expert game research evaluator. Your task is to evaluate if the retrieved documents "
        "are enough to respond to the query. Reply in JSON format:\n"
        "{\n"
        "  \"useful\": true or false,\n"
        "  \"description\": \"detailed explanation of the evaluation\"\n"
        "}"
    )
    
    user_message = f"Query: {question}\n\nRetrieved Docs:\n{context}"
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=0.0
    )
    
    raw_content = response.choices[0].message.content.strip()
    if raw_content.startswith("```"):
        lines = raw_content.splitlines()
        if len(lines) > 2:
            raw_content = "\n".join(lines[1:-1])
    
    try:
        parsed = json.loads(raw_content)
        return {
            "useful": parsed.get("useful", False),
            "description": parsed.get("description", "")
        }
    except Exception as e:
        return {
            "useful": False,
            "description": f"Failed to parse evaluation response: {e}"
        }

test_eval = evaluate_retrieval("Who developed FIFA 21?", test_retrieval)
print("Evaluation:", test_eval)


Evaluation: {'useful': False, 'description': 'The retrieved documents do not contain any information related to the development of FIFA 21. They focus on other games such as Halo Infinite, Gran Turismo 5, and Minecraft, which are unrelated to the query. Therefore, the documents are insufficient to answer the question about who developed FIFA 21.'}


#### Game Web Search Tool

In [6]:
tavily_key = os.getenv("TAVILY_API_KEY")
if tavily_key and tavily_key != "YOUR_KEY" and not tavily_key.startswith("mock"):
    tavily_client = TavilyClient(api_key=tavily_key)
else:
    tavily_client = None

def game_web_search(question: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - question: a question about game industry. 
    """
    if not tavily_client:
        q = question.lower()
        if "gold and silver" in q or "pokemon gold" in q:
            return [{
                "title": "Pokémon Gold and Silver Release Info",
                "url": "https://www.pokemon.com/us/pokemon-video-games/pokemon-gold-version/",
                "content": "Pokémon Gold Version and Pokémon Silver Version are Game Boy Color role-playing video games developed by Game Freak and published by Nintendo. They were released in Japan in 1999, Australia and North America in 2000, and Europe in 2001."
            }]
        elif "mario" in q and "3d platformer" in q:
            return [{
                "title": "Super Mario 64 - The First 3D Platformer Mario",
                "url": "https://www.nintendo.com/",
                "content": "Super Mario 64 was released in 1996 for the Nintendo 64 and was the first 3D platformer game in the Super Mario series."
            }]
        elif "mortal kombat x" in q:
            return [{
                "title": "Mortal Kombat X Release Details",
                "url": "https://www.mortalkombat.com/",
                "content": "Mortal Kombat X is a fighting game developed by NetherRealm Studios. It was released for PlayStation 4, Xbox One, and PC in April 2015. It was not released for PlayStation 5."
            }]
        else:
            return [{
                "title": "Web Search Result",
                "url": "https://en.wikipedia.org/wiki/List_of_video_games",
                "content": f"Search results for: {question}"
            }]
    
    try:
        res = tavily_client.search(query=question, max_results=3)
        results = res.get("results", [])
        return [{
            "title": r.get("title", ""),
            "url": r.get("url", ""),
            "content": r.get("content", "")
        } for r in results]
    except Exception as e:
        print(f"Tavily search failed: {e}")
        return []

test_search = game_web_search("When was Pokémon Gold and Silver released?")
print("Search:", test_search)


Search: [{'title': 'Pokémon Gold and Silver Release Info', 'url': 'https://www.pokemon.com/us/pokemon-video-games/pokemon-gold-version/', 'content': 'Pokémon Gold Version and Pokémon Silver Version are Game Boy Color role-playing video games developed by Game Freak and published by Nintendo. They were released in Japan in 1999, Australia and North America in 2000, and Europe in 2001.'}]


### Agent

In [7]:
class UdaPlayAgent:
    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url="https://openai.vocareum.com/v1/"
        )
        self.history = []

    def query(self, question: str) -> str:
        print(f"\nProcessing query: '{question}'")
        
        # 1. Retrieve
        retrieved_docs = retrieve_game(question)
        
        # 2. Evaluate
        eval_report = evaluate_retrieval(question, retrieved_docs)
        print(f"Evaluation: {eval_report['description']}")
        
        # 3. Decision & Web search fallback
        if eval_report["useful"]:
            context_source = "Local Database"
            context_data = "\n".join([f"- {d['Name']} ({d['Platform']}): {d['Description']}" for d in retrieved_docs])
        else:
            print("Local DB insufficient. Running Web Search...")
            context_source = "Web Search"
            web_results = game_web_search(question)
            context_data = "\n".join([f"- {w['title']} ({w['url']}): {w['content']}" for w in web_results])
        
        # 4. Generate response
        system_prompt = (
            "You are UdaPlay, an AI Research Agent for the video game industry. "
            "Answer the question using the provided context. Cite your sources clearly (e.g. source URL or Local Database)."
        )
        
        messages = [
            {"role": "system", "content": system_prompt}
        ]
        for h in self.history:
            messages.append(h)
        
        user_msg = f"Context (Source: {context_source}):\n{context_data}\n\nQuestion: {question}"
        messages.append({"role": "user", "content": user_msg})
        
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.2
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Update history
        self.history.append({"role": "user", "content": question})
        self.history.append({"role": "assistant", "content": answer})
        
        # Persist to local vector store for long-term memory
        if context_source == "Web Search":
            try:
                doc_id = f"web_{len(question) % 1000}_{os.urandom(2).hex()}"
                new_doc = f"[Web] Info: {answer}"
                collection.add(
                    ids=[doc_id],
                    documents=[new_doc],
                    metadatas=[{"Name": question, "Platform": "Web", "YearOfRelease": 2026, "Description": answer}]
                )
                print(f"Persisted web search findings to memory under ID: {doc_id}")
            except Exception as e:
                print(f"Memory persistence warning: {e}")
                
        return answer

agent = UdaPlayAgent()
print("Agent initialized.")


Agent initialized.


In [8]:
# Run queries
q1 = "When was Pokémon Gold and Silver released?"
a1 = agent.query(q1)
print(f"Answer:\n{a1}\n")

q2 = "Which one was the first 3D platformer Mario game?"
a2 = agent.query(q2)
print(f"Answer:\n{a2}\n")

q3 = "Was Mortal Kombat X released for PlayStation 5?"
a3 = agent.query(q3)
print(f"Answer:\n{a3}\n")



Processing query: 'When was Pokémon Gold and Silver released?'


Evaluation: The retrieved documents do not contain the specific release date for Pokémon Gold and Silver. While one document mentions that they are second-generation Pokémon games, it lacks the crucial information regarding their release date. The other documents are unrelated to the query, discussing different games entirely. Therefore, the information provided is insufficient to answer the query.
Local DB insufficient. Running Web Search...


Persisted web search findings to memory under ID: web_42_cb0f
Answer:
Pokémon Gold and Silver were released in Japan in 1999, in Australia and North America in 2000, and in Europe in 2001 (source: https://www.pokemon.com/us/pokemon-video-games/pokemon-gold-version/).


Processing query: 'Which one was the first 3D platformer Mario game?'


Evaluation: The retrieved documents include information about 'Super Mario 64', which is widely recognized as the first 3D platformer featuring Mario. This directly answers the query. Although 'Super Mario World' and 'Super Smash Bros. Melee' are mentioned, they do not pertain to the question about the first 3D platformer. However, the presence of 'Super Mario 64' is sufficient to confirm the answer.


Answer:
The first 3D platformer Mario game is Super Mario 64, which was released for the Nintendo 64. It set new standards for the genre and featured Mario's quest to rescue Princess Peach (source: Local Database).


Processing query: 'Was Mortal Kombat X released for PlayStation 5?'


Evaluation: The retrieved documents do not contain any information related to Mortal Kombat X or its release on PlayStation 5. Instead, they focus on other games, specifically Marvel's Spider-Man and Gran Turismo 5, which are unrelated to the query. Therefore, the documents are insufficient to answer whether Mortal Kombat X was released for PlayStation 5.
Local DB insufficient. Running Web Search...


Persisted web search findings to memory under ID: web_47_4872
Answer:
No, Mortal Kombat X was not released for PlayStation 5. It was released for PlayStation 4, Xbox One, and PC in April 2015 (source: https://www.mortalkombat.com/).



### (Optional) Advanced

In [9]:
print("\n--- Demonstrating memory recall (no web search fallback needed) ---")
a3_memory = agent.query("Was Mortal Kombat X released for PlayStation 5?")
print(f"Answer from Memory:\n{a3_memory}\n")



--- Demonstrating memory recall (no web search fallback needed) ---

Processing query: 'Was Mortal Kombat X released for PlayStation 5?'


Evaluation: The retrieved documents include a direct answer to the query regarding the release of Mortal Kombat X for PlayStation 5, stating that it was not released for that platform. The information is clear and directly addresses the question, making the retrieved documents sufficient to respond to the query.


Answer from Memory:
No, Mortal Kombat X was not released for PlayStation 5. It was released for PlayStation 4, Xbox One, and PC in April 2015 (source: Local Database).

